In [16]:
import asyncio
import sys
import time
import re

In [6]:
class Logable:
    
    def __init__(self, name, verbose):
        self.name = name
        self.verbose = verbose

    def log(self, msg):
        if self.verbose:
            print(f"[{self.name}] {msg}", file=logf, flush=True)

In [2]:
class Accessoire(Logable):

    def __init__(self, name, verbose=True):
        Logable.__init__(self, name, verbose)
        self.queue = asyncio.Queue()

In [7]:
class Pic(Accessoire):

    async def embrocher(self, postit):
        await self.queue.put(postit)

        self.log(
            f"post-it '{postit}' embroché, "
            f"{self.queue.qsize()} post-it(s) à traiter"
        )

    async def liberer(self):
        postit = await self.queue.get()

        self.log(
            f"post-it '{postit}' libéré, "
            f"{self.queue.qsize()} post-it(s) à traiter"
        )

        return postit

In [8]:
class Bar(Accessoire):

    async def recevoir(self, commande):
        await self.queue.put(commande)

        self.log(
            f"'{commande}' posée, "
            f"{self.queue.qsize()} commande(s) à servir"
        )

    async def evacuer(self):
        commande = await self.queue.get()

        self.log(
            f"'{commande}' évacuée, "
            f"{self.queue.qsize()} commande(s) à servir"
        )

        return commande

In [9]:
class Clients:

    def __init__(self, fname):
        commandes = []
        start = time.time()

        fmt = re.compile(r"(\d+)\s+(.*)")

        with open(fname, "r", encoding="utf-8") as f:
            for line in f:
                found = fmt.search(line)

                if found:
                    when = int(found.group(1))
                    what = found.group(2)

                    commandes.append(
                        (start + when, what.split(","))
                    )

        self.commandes = commandes[::-1]

    async def run(self, serveur):

        while self.commandes:

            when, commande = self.commandes[-1]

            attente = when - time.time()

            if attente > 0:
                await asyncio.sleep(attente)

            self.commandes.pop()

            await serveur.prendre_commande(commande)

        # Signal de fin envoyé au pic
        await serveur.pic.embrocher(None)

In [10]:
class Clients:

    def __init__(self, fname):
        commandes = []
        start = time.time()

        fmt = re.compile(r"(\d+)\s+(.*)")

        with open(fname, "r", encoding="utf-8") as f:
            for line in f:
                found = fmt.search(line)

                if found:
                    when = int(found.group(1))
                    what = found.group(2)

                    commandes.append(
                        (start + when, what.split(","))
                    )

        self.commandes = commandes[::-1]

    async def run(self, serveur):

        while self.commandes:

            when, commande = self.commandes[-1]

            attente = when - time.time()

            if attente > 0:
                await asyncio.sleep(attente)

            self.commandes.pop()

            await serveur.prendre_commande(commande)

        # Signal de fin envoyé au pic
        await serveur.pic.embrocher(None)

In [22]:
class Employe(Logable):

    def __init__(self, pic, bar, clients, name, verbose=True):
        
        Logable.__init__(self, name, verbose)

        self.pic = pic
        self.bar = bar
        self.clients = clients

        self.step = 0

In [24]:
class Serveur(Employe):

    def __init__(self, pic, bar, clients, name="alice", verbose=True):
         super().__init__(pic, bar, clients, name, verbose)
         self.lock = asyncio.Lock()

    async def prendre_commande(self, commande):

        await self.lock.acquire()

        self.log("prêt pour prendre une nouvelle commande...")
        self.log(f"j'ai la commande '{commande}'")
        self.log(f"j'écris sur le post-it '{commande}'")

        await self.pic.embrocher(commande)

    async def servir(self):

        commande = await self.bar.evacuer()

        if commande is None:
            return False

        self.log(f"j'apporte la commande '{commande}'")

        for cocktail in commande:
            self.log(f"je sers '{cocktail}'")
            await asyncio.sleep(0)

        self.lock.release()

        return True

    async def run(self):

        while True:
            commande = await self.servir()

            if not commande:
                break

In [13]:
class Bariste(Employe):

    async def preparer(self):

        commande = await self.pic.liberer()

        # Signal de fin
        if commande is None:
            await self.bar.recevoir(None)
            return None

        self.log(f"je commence la fabrication de '{commande}'")

        for cocktail in commande:

            self.log(f"je prépare '{cocktail}'")

            # Étape 3 : coopération
            await asyncio.sleep(0)

        await self.bar.recevoir(commande)

        self.log(f"la commande {commande} est prête")

        return commande

    async def run(self):

        while True:

            commande = await self.preparer()

            if commande is None:
                break

In [14]:
async def main():

    le_pic = Pic(name="le_pic", verbose=False)
    le_bar = Bar(name="le_bar", verbose=False)

    les_clients = Clients("lundi")

    bob = Bariste(
        le_pic, le_bar, les_clients,
        name="bob", verbose=True
    )

    alice = Serveur(
        le_pic, le_bar, les_clients,
        name="alice", verbose=True
    )

    await asyncio.gather(
        les_clients.run(alice),
        bob.run(),
        alice.run()
    )

In [26]:
logfile = "borabora.log"

logf = open(logfile, "w", encoding="utf-8")

print("\n---", file=logf, flush=True)

await main()

logf.close()